In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

In [2]:
df = pd.read_csv("../data/income_data.csv")


Limpieza básica:

In [11]:
df.columns = df.columns.str.strip()
df = df.drop_duplicates().copy()
df = df.replace("?", np.nan)

In [12]:
df["income"] = df["income"].str.strip()


Features eingeneering básicos (ayuda con ia )

In [13]:
df["has_capital_gain"] = (df["capital-gain"] > 0).astype(int)
df["has_capital_loss"] = (df["capital-loss"] > 0).astype(int)

Hago este feature engineering para crear variables más útiles a partir de las originales y ayudar al modelo a captar mejor patrones relevantes

Separo x/y:

In [14]:
X = df.drop(columns=["income"])
y = df["income"].map({">50K": 1, "<=50K": 0})

Compruebo que las columnas existen

In [15]:
print("Columnas de X:")
print(X.columns.tolist())

print("\nComprobación:")
print("has_capital_gain" in X.columns)
print("has_capital_loss" in X.columns)

Columnas de X:
['age', 'workclass', 'fnlwgt', 'education', 'educational-num', 'marital-status', 'occupation', 'relationship', 'race', 'gender', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'has_capital_gain', 'has_capital_loss']

Comprobación:
True
True


Columnas:

In [16]:
num_cols = [
    "age", "fnlwgt", "educational-num",
    "capital-gain", "capital-loss", "hours-per-week",
    "has_capital_gain", "has_capital_loss"
]

cat_cols = [
    "workclass", "education", "marital-status", "occupation",
    "relationship", "race", "gender", "native-country"
]


Defino las columnas numéricas y categóricas para aplicar a cada tipo de variable el preprocesamiento que necesita

Split

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Shape X_train:", X_train.shape)
print("Shape X_test:", X_test.shape)
print("Shape y_train:", y_train.shape)
print("Shape y_test:", y_test.shape)

Shape X_train: (39032, 16)
Shape X_test: (9758, 16)
Shape y_train: (39032,)
Shape y_test: (9758,)


He hecho el split porque no tendría sentido evaluar el modelo con los mismos datos que ha usado para aprender. Quiero comprobar si funciona bien con información nueva, y por eso separo train y test

Compruebo columnas de x_train

In [18]:
print("Columnas X_train:")
print(X_train.columns.tolist())

Columnas X_train:
['age', 'workclass', 'fnlwgt', 'education', 'educational-num', 'marital-status', 'occupation', 'relationship', 'race', 'gender', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'has_capital_gain', 'has_capital_loss']


Creo transformers

In [19]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first"))
])

Creo el processor

In [20]:
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
])

He hecho este bloque de preprocessing para preparar el dataset antes de entrenar el modelo. Aquí aplico la imputación de nulos, la codificación de las variables categóricas y el escalado de las numéricas. Me parece importante porque cada tipo de variable necesita un tratamiento distinto y así consigo que el modelo reciba los datos de una forma más limpia y ordenada

guardo archivos

In [21]:
os.makedirs("../data/models", exist_ok=True)

joblib.dump(preprocessor, "../data/models/preprocessor.pkl")
joblib.dump((X_train, X_test, y_train, y_test), "../data/models/splits.pkl")

print("Archivos guardados correctamente.")

Archivos guardados correctamente.


Comprobacion final

In [22]:
print("Número de columnas en X_train:", X_train.shape[1])
print("¿Está has_capital_gain?", "has_capital_gain" in X_train.columns)
print("¿Está has_capital_loss?", "has_capital_loss" in X_train.columns)

Número de columnas en X_train: 16
¿Está has_capital_gain? True
¿Está has_capital_loss? True


Conclusiones del preprocessing: Como resultado del preprocessing, el dataset queda limpio y preparado para el modelado. Se han tratado los valores faltantes, separado las variables numéricas y categóricas y aplicado las transformaciones necesarias para que el modelo pueda entrenarse correctamente.
